# OECD FDI Income: Panel and Time-Structured Models with Feature Creation

This notebook moves from general regression to panel/time-structured modeling.

Important design note:

- The dataset contains only 2023 and 2024.
- That means a classical stand-alone time-series model is not reliable here.
- A short-panel formulation is more appropriate than a pure time-series formulation.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display, Markdown


C:\Users\Yusuf\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent

data_path = project_root / 'data' / 'processed' / 'OECD_FDI_Except_Pointless.csv'
df = pd.read_csv(data_path, sep=';', encoding='latin1', engine='python', on_bad_lines='skip')
df['OBS_VALUE_NUM'] = pd.to_numeric(
    df['OBS_VALUE'].astype(str).str.replace('.', '', regex=False),
    errors='coerce'
)
df['obs_missing_flag'] = df['OBS_VALUE_NUM'].isna().astype(int)
df['obs_value_imputed'] = df['OBS_VALUE_NUM'].fillna(
    df.groupby(['REF_AREA', 'ACTIVITY', 'TIME_PERIOD'])['OBS_VALUE_NUM'].transform('median')
).fillna(df['OBS_VALUE_NUM'].median())
df['obs_value_signed_log'] = np.sign(df['obs_value_imputed']) * np.log1p(np.abs(df['obs_value_imputed']))
df['panel_id'] = (
    df['REF_AREA'].astype(str) + '__' +
    df['ACTIVITY'].astype(str) + '__' +
    df['TYPE_ENTITY'].astype(str) + '__' +
    df['MEASURE_PRINCIPLE'].astype(str)
)
df[['panel_id', 'TIME_PERIOD', 'obs_value_signed_log']].head()


,panel_id,TIME_PERIOD,obs_value_signed_log
0,USA__A_B__RSP__DO,2024,9.450459
1,AUS__J__ALL__DI,2023,30.596330
2,AUS__D__ALL__DI,2023,30.627683
3,LUX__C29_30__ROU__DO,2024,32.209225
4,AUS__J__ROU__DO,2023,30.596330


## Balanced Short Panels and New Panel Variables


In [3]:
panel_counts = df.groupby('panel_id')['TIME_PERIOD'].nunique()
balanced_panels = panel_counts[panel_counts == 2].index
panel_df = df[df['panel_id'].isin(balanced_panels)].copy()
panel_df = panel_df.sort_values(['panel_id', 'TIME_PERIOD']).copy()

panel_df['panel_mean_signed_log'] = panel_df.groupby('panel_id')['obs_value_signed_log'].transform('mean')
panel_df['panel_within_centered_signed_log'] = panel_df['obs_value_signed_log'] - panel_df['panel_mean_signed_log']
panel_df['panel_lag_signed_log'] = panel_df.groupby('panel_id')['obs_value_signed_log'].shift(1)
panel_df['panel_diff_signed_log'] = panel_df.groupby('panel_id')['obs_value_signed_log'].diff()
panel_df['panel_growth_ratio_raw'] = panel_df.groupby('panel_id')['obs_value_imputed'].pct_change()
panel_df['is_balanced_panel'] = 1

display(Markdown(f'**Balanced short panels used:** {panel_df['panel_id'].nunique():,}'))
panel_df[['panel_id', 'TIME_PERIOD', 'panel_mean_signed_log', 'panel_within_centered_signed_log', 'panel_lag_signed_log', 'panel_diff_signed_log']].head()


**Balanced short panels used:** 4,674

,panel_id,TIME_PERIOD,panel_mean_signed_log,panel_within_centered_signed_log,panel_lag_signed_log,panel_diff_signed_log
41,AUS__A_B__ALL__DI,2023,24.567743,-6.239835,NaN,NaN
157,AUS__A_B__ALL__DI,2024,24.567743,6.239835,18.327907,12.479671
82,AUS__A_B__ALL__DO,2023,24.567743,-6.239835,NaN,NaN
83,AUS__A_B__ALL__DO,2024,24.567743,6.239835,18.327907,12.479671
231,AUS__A_B__ROU__DI,2023,24.567743,-6.239835,NaN,NaN


### Comment

Because there are only two years, the meaningful dynamic quantities are differences, lags, and within-panel deviations rather than a conventional long-run autoregressive time series.


## Panel Models

Three models are estimated:

- pooled OLS
- fixed-effects style OLS with panel dummies
- random-intercept mixed model


In [4]:
panel_model_df = panel_df.dropna(subset=['obs_value_signed_log']).copy()

pooled_formula = 'obs_value_signed_log ~ C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD + obs_missing_flag'
fe_formula = 'obs_value_signed_log ~ C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + C(TIME_PERIOD) + C(panel_id)'
re_formula = 'obs_value_signed_log ~ C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD + obs_missing_flag'

pooled_model = smf.ols(pooled_formula, data=panel_model_df).fit(cov_type='HC3')
fe_model = smf.ols(fe_formula, data=panel_model_df).fit(cov_type='HC3')
re_model = smf.mixedlm(re_formula, data=panel_model_df, groups=panel_model_df['panel_id']).fit(reml=False, method='lbfgs')

print(pooled_model.summary())


                             OLS Regression Results                             
Dep. Variable:     obs_value_signed_log   R-squared:                       0.120
Model:                              OLS   Adj. R-squared:                  0.119
Method:                   Least Squares   F-statistic:                     389.1
Date:                  Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                          19:13:01   Log-Likelihood:                -39158.
No. Observations:                  9348   AIC:                         7.833e+04
Df Residuals:                      9342   BIC:                         7.837e+04
Df Model:                             5                                         
Covariance Type:                    HC3                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------


In [ ]:
print(fe_model.summary())


In [ ]:
print(re_model.summary())


## Model Choice and Final Panel Variables


In [ ]:
model_compare = pd.DataFrame([
    {'model': 'Pooled OLS', 'aic': pooled_model.aic, 'bic': pooled_model.bic},
    {'model': 'Fixed Effects OLS', 'aic': fe_model.aic, 'bic': fe_model.bic},
    {'model': 'Random Intercept MixedLM', 'aic': re_model.aic, 'bic': re_model.bic},
]).sort_values('aic')
model_compare


In [ ]:
primary_panel_model_name = model_compare.iloc[0]['model']
if primary_panel_model_name == 'Pooled OLS':
    panel_df['panel_primary_fitted'] = pooled_model.predict(panel_df)
elif primary_panel_model_name == 'Fixed Effects OLS':
    panel_df['panel_primary_fitted'] = fe_model.predict(panel_df)
else:
    panel_df['panel_primary_fitted'] = re_model.predict(panel_df)

panel_df['panel_primary_residual'] = panel_df['obs_value_signed_log'] - panel_df['panel_primary_fitted']
panel_df['panel_primary_fitted_rawscale'] = np.sign(panel_df['panel_primary_fitted']) * np.expm1(np.abs(panel_df['panel_primary_fitted']))

comment = (
    f'The preferred short-panel specification by information criteria is: {primary_panel_model_name}. '
    'A pure time-series model is not treated as primary because two annual periods are insufficient for a stable stand-alone time-series structure.'
)
display(Markdown('### Comment\n\n' + comment))
panel_df[['panel_primary_fitted', 'panel_primary_residual', 'panel_primary_fitted_rawscale']].head()
